# CADT 实验结果汇总

In [1]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

%matplotlib inline
sns.set_style('whitegrid')

In [2]:
def parse_results_file(filepath):
    '''解析单个 results.txt 文件，提取关键指标'''
    with open(filepath, 'r') as f:
        content = f.read()

    result = {}

    timestamp_match = re.search(r'Timestamp: (\d{8}_\d{6})', content)
    if timestamp_match:
        result['timestamp'] = timestamp_match.group(1)

    split_mode_match = re.search(r'Split Mode: (\w+)', content)
    if split_mode_match:
        result['split_mode'] = split_mode_match.group(1)

    target_subject_match = re.search(r'Target Subject: (\d+)', content)
    if target_subject_match:
        result['target_subject'] = int(target_subject_match.group(1))
    else:
        result['target_subject'] = 'all'

    best_epoch_match = re.search(r'Best Epoch: (\d+)', content)
    if best_epoch_match:
        result['best_epoch'] = int(best_epoch_match.group(1))

    n_source_domains_match = re.search(r'Number of Source Domains: (\d+)', content)
    if n_source_domains_match:
        result['n_source_domains'] = int(n_source_domains_match.group(1))

    val_mse_match = re.search(r'Val MSE: ([\d.]+)', content)
    if val_mse_match:
        result['val_mse'] = float(val_mse_match.group(1))

    val_mae_match = re.search(r'Val MAE: ([\d.]+)', content)
    if val_mae_match:
        result['val_mae'] = float(val_mae_match.group(1))

    val_rmse_match = re.search(r'Val RMSE: ([\d.]+)', content)
    if val_rmse_match:
        result['val_rmse'] = float(val_rmse_match.group(1))

    val_r2_match = re.search(r'Val R2: ([\d.-]+)', content)
    if val_r2_match:
        result['val_r2'] = float(val_r2_match.group(1))

    val_avg_r2_match = re.search(r'Val Avg R2 \(per-dim\): ([\d.-]+)', content)
    if val_avg_r2_match:
        result['val_avg_r2'] = float(val_avg_r2_match.group(1))

    val_pearson_match = re.search(r'Val Pearson: ([\d.-]+)', content)
    if val_pearson_match:
        result['val_pearson'] = float(val_pearson_match.group(1))

    val_spearman_match = re.search(r'Val Spearman: ([\d.-]+)', content)
    if val_spearman_match:
        result['val_spearman'] = float(val_spearman_match.group(1))

    test_mse_match = re.search(r'Test MSE: ([\d.]+)', content)
    if test_mse_match:
        result['test_mse'] = float(test_mse_match.group(1))

    test_mae_match = re.search(r'Test MAE: ([\d.]+)', content)
    if test_mae_match:
        result['test_mae'] = float(test_mae_match.group(1))

    test_rmse_match = re.search(r'Test RMSE: ([\d.]+)', content)
    if test_rmse_match:
        result['test_rmse'] = float(test_rmse_match.group(1))

    test_r2_match = re.search(r'Test R2: ([\d.-]+)', content)
    if test_r2_match:
        result['test_r2'] = float(test_r2_match.group(1))

    test_avg_r2_match = re.search(r'Test Avg R2 \(per-dim\): ([\d.-]+)', content)
    if test_avg_r2_match:
        result['test_avg_r2'] = float(test_avg_r2_match.group(1))

    test_pearson_match = re.search(r'Test Pearson: ([\d.-]+)', content)
    if test_pearson_match:
        result['test_pearson'] = float(test_pearson_match.group(1))

    test_spearman_match = re.search(r'Test Spearman: ([\d.-]+)', content)
    if test_spearman_match:
        result['test_spearman'] = float(test_spearman_match.group(1))

    per_emotion_section = re.search(r'--- Per-Emotion Metrics ---.*?(?=\n\n|\nModel Saved:|$)', content, re.DOTALL)
    if per_emotion_section:
        emotion_lines = per_emotion_section.group(0).split('\n')[2:]
        emotion_metrics = {}
        for line in emotion_lines:
            if line.strip() and not line.strip().startswith('---') and not line.strip().startswith('Average'):
                parts = re.split(r'\s+', line.strip())
                if len(parts) >= 6:
                    # 新格式: Emotion MSE MAE R2 Pearson Spearman
                    emotion = parts[0]
                    emotion_metrics[emotion] = {}
                    emotion_metrics[emotion]['mse'] = float(parts[1])
                    emotion_metrics[emotion]['mae'] = float(parts[2])
                    emotion_metrics[emotion]['r2'] = float(parts[3])
                    emotion_metrics[emotion]['pearson'] = float(parts[4])
                    emotion_metrics[emotion]['spearman'] = float(parts[5])
                elif len(parts) >= 5:
                    # 兼容旧格式: Emotion MSE MAE Pearson Spearman
                    emotion = parts[0]
                    emotion_metrics[emotion] = {}
                    emotion_metrics[emotion]['mse'] = float(parts[1])
                    emotion_metrics[emotion]['mae'] = float(parts[2])
                    emotion_metrics[emotion]['pearson'] = float(parts[3])
                    emotion_metrics[emotion]['spearman'] = float(parts[4])
        result['per_emotion'] = emotion_metrics

    return result

In [3]:
def parse_params_file(filepath):
    '''解析 params.txt 文件，提取训练参数'''
    with open(filepath, 'r') as f:
        content = f.read()

    params = {}
    params_match = re.findall(r'(\w+): ([\d.]+)', content)
    for key, value in params_match:
        try:
            params[key] = int(value)
        except ValueError:
            try:
                params[key] = float(value)
            except ValueError:
                params[key] = value
    return params

In [4]:
def load_all_experiments(results_dir='/root/zjm/cadt/results4'):
    '''加载 results_dir 下所有实验结果'''
    experiments = []
    for entry in os.listdir(results_dir):
        entry_path = os.path.join(results_dir, entry)
        if os.path.isdir(entry_path) and 'emotion' in entry:
            results_file = os.path.join(entry_path, 'results.txt')
            params_file = os.path.join(entry_path, 'params.txt')
            if os.path.exists(results_file):
                result = parse_results_file(results_file)
                result['directory'] = entry
                if os.path.exists(params_file):
                    params = parse_params_file(params_file)
                    result.update(params)
                experiments.append(result)
    return experiments

In [5]:
# 加载所有实验
experiments = load_all_experiments()
print(f'共加载 {len(experiments)} 个实验')

# 转换为 DataFrame
df = pd.DataFrame(experiments)
df['datetime'] = pd.to_datetime(df['timestamp'], format='%Y%m%d_%H%M%S')
df = df.sort_values('datetime').reset_index(drop=True)

# 评价指标列表
metrics = ['val_mse', 'val_mae', 'val_rmse', 'val_r2', 'val_avg_r2', 'val_pearson', 'val_spearman',
           'test_mse', 'test_mae', 'test_rmse', 'test_r2', 'test_avg_r2', 'test_pearson', 'test_spearman']

# 显示基本信息
df[['timestamp', 'split_mode', 'target_subject', 'best_epoch', 'n_source_domains', 'val_mse', 'val_r2', 'val_avg_r2', 'test_mse', 'test_r2', 'test_avg_r2']]

FileNotFoundError: [Errno 2] No such file or directory: '/root/zjm/cadt/results4'

## 多Seed实验汇总（均值±标准差）

In [ ]:
def format_mean_std(mean, std, decimals=4):
    '''格式化均值±标准差'''
    return f"{mean:.{decimals}f} ± {std:.{decimals}f}"

# 按 split_mode 和 target_subject 分组
grouped = df.groupby(['split_mode', 'target_subject'])

# 计算均值、标准差和统计显著性
summary_list = []

for (split_mode, subject), group in grouped:
    row = {
        'split_mode': split_mode,
        'target_subject': subject,
        'n_seeds': len(group),
        'timestamps': ', '.join(group['timestamp'].tolist())
    }
    
    for metric in metrics:
        if metric in group.columns:
            values = group[metric].dropna().values
            if len(values) > 0:
                row[f'{metric}_mean'] = np.mean(values)
                row[f'{metric}_std'] = np.std(values, ddof=1)
                row[f'{metric}_mean_std'] = format_mean_std(np.mean(values), np.std(values, ddof=1))
                
                # 计算95%置信区间
                if len(values) >= 2:
                    ci_low, ci_high = stats.t.interval(0.95, len(values)-1, loc=np.mean(values), scale=stats.sem(values))
                    row[f'{metric}_ci_95'] = f"[{ci_low:.4f}, {ci_high:.4f}]"
                
                # 不同seed间差异的显著性检验
                if len(values) >= 2:
                    # 使用单样本t检验: 检验seed均值是否显著偏离0（零假设）
                    # 注意: 这里不是直接比较seed，而是检验结果是否显著非零
                    if len(values) >= 2 and np.std(values, ddof=1) > 0:
                        t_stat, p_one = stats.ttest_1samp(values, 0)
                        row[f'{metric}_t_stat'] = t_stat
                        row[f'{metric}_p_value'] = p_one
                        row[f'{metric}_significant'] = '**' if p_one < 0.01 else '*' if p_one < 0.05 else 'ns'
                    # Levene方差齐性检验需要至少两组独立数据
                    # 同一受试者内无法做Levene检验（只有一组seed数据）
                    # 如需Levene检验，请使用跨受试者对比的方差分析
    
    summary_list.append(row)

summary_df = pd.DataFrame(summary_list)

# 显示汇总表格 - 分为验证集和测试集两个表格
val_metrics = [m for m in metrics if m.startswith('val_')]
test_metrics = [m for m in metrics if m.startswith('test_')]

print('=' * 80)
print('验证集结果 (Validation Set)')
print('=' * 80)
val_cols = ['split_mode', 'target_subject', 'n_seeds'] + [f'{m}_mean_std' for m in val_metrics]
display(summary_df[val_cols])

print('=' * 80)
print('测试集结果 (Test Set)')
print('=' * 80)
test_cols = ['split_mode', 'target_subject', 'n_seeds'] + [f'{m}_mean_std' for m in test_metrics]
display(summary_df[test_cols])

## 统计显著性检验

### 1. 不同受试者的差异显著性（ANOVA）
- 零假设 H₀ ：所有受试者的指标均值相同（μ₁ = μ₂ = ... = μₖ）
- 备择假设 H₁ ：至少有一个受试者的均值与其他受试者不同

In [ ]:
def anova_test(data, group_col, value_col):
    '''单因素方差分析'''
    groups = []
    for name, group in data.groupby(group_col):
        groups.append(group[value_col].dropna().values)
    
    if len(groups) >= 2:
        f_stat, p_value = stats.f_oneway(*groups)
        return f_stat, p_value
    return np.nan, np.nan

# 对每个指标进行ANOVA检验
anova_results = []

# 只分析 cross_day 模式
cross_day_df = df[df['split_mode'] == 'cross_day'].copy()

for metric in metrics:
    if metric in cross_day_df.columns:
        # 按target_subject分组，每个subject作为一组
        f_stat, p_value = anova_test(cross_day_df, 'target_subject', metric)
        anova_results.append({
            'metric': metric,
            'f_stat': f_stat,
            'p_value': p_value,
            'significant': '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
            })

anova_df = pd.DataFrame(anova_results)
anova_df.style.format({
    'f_stat': '{:.4f}',
    'p_value': '{:.6f}'
}).set_table_styles([{
    'selector': 'th',
    'props': [('font-size', '11px'), ('text-align', 'center')]
}, {
    'selector': 'td',
    'props': [('font-size', '11px'), ('text-align', 'center')]
}])

### 2. 事后检验：Tukey HSD 检验

当ANOVA结果显著时（p < 0.05），使用Tukey HSD检验进行两两比较，找出哪些受试者之间存在显著差异(reject=True)。

In [ ]:
# Tukey HSD 事后检验
for metric in ['val_r2', 'test_r2', 'val_mse', 'test_mse']:
    if metric in cross_day_df.columns:
        data = cross_day_df.dropna(subset=[metric]).copy()
        data['subject_str'] = data['target_subject'].astype(str)
        
        tukey = pairwise_tukeyhsd(endog=data[metric], groups=data['subject_str'], alpha=0.05)
        
        print(f'\n=== {metric} - Tukey HSD 检验 ===')
        print(tukey.summary())
        
        # 可视化
        fig = tukey.plot_simultaneous()
        plt.title(f'Tukey HSD - {metric}')
        plt.show()

## 可视化对比

In [ ]:
colors = {'cross_subject': '#1f77b4', 'cross_day': '#ff7f0e', 'within_subject_day': '#2ca02c'}

plot_data = summary_df.copy()
plot_data['subject_label'] = plot_data.apply(
    lambda row: f"{row['split_mode']}{'_subj' + str(row['target_subject']) if row['target_subject'] != 'all' else ''}",
    axis=1
)

plt.figure(figsize=(16, 6))

# Val R2
plt.subplot(1, 2, 1)
ax = sns.barplot(data=plot_data, x='subject_label', y='val_r2_mean', hue='split_mode', palette=colors)
for i, (_, row) in enumerate(plot_data.iterrows()):
    ax.errorbar(i, row['val_r2_mean'], yerr=row['val_r2_std'], fmt='none', c='black', capsize=5)
plt.title('Validation R2 Score (mean ± std)')
plt.xticks(rotation=45)
plt.ylabel('R2')

# Test R2
plt.subplot(1, 2, 2)
ax = sns.barplot(data=plot_data, x='subject_label', y='test_r2_mean', hue='split_mode', palette=colors)
for i, (_, row) in enumerate(plot_data.iterrows()):
    ax.errorbar(i, row['test_r2_mean'], yerr=row['test_r2_std'], fmt='none', c='black', capsize=5)
plt.title('Test R2 Score (mean ± std)')
plt.xticks(rotation=45)
plt.ylabel('R2')

plt.tight_layout()
plt.show()

## 原始实验记录列表

In [ ]:
details_df = df[['timestamp', 'split_mode', 'target_subject', 'n_source_domains', 'best_epoch', 
                 'val_mse', 'val_mae', 'val_rmse', 'val_r2', 'val_pearson', 'val_spearman',
                 'test_mse', 'test_mae', 'test_rmse', 'test_r2', 'test_pearson', 'test_spearman',
                 'batch_size', 'n_epoch']]

details_df.style.format({
    'val_mse': '{:.6f}',
    'val_mae': '{:.6f}',
    'val_rmse': '{:.6f}',
    'val_r2': '{:.6f}',
    'val_pearson': '{:.6f}',
    'val_spearman': '{:.6f}',
    'test_mse': '{:.6f}',
    'test_mae': '{:.6f}',
    'test_rmse': '{:.6f}',
    'test_r2': '{:.6f}',
    'test_pearson': '{:.6f}',
    'test_spearman': '{:.6f}'
}).set_table_styles([{
    'selector': 'th',
    'props': [('font-size', '10px'), ('text-align', 'center')]
}, {
    'selector': 'td',
    'props': [('font-size', '10px'), ('text-align', 'center')]
}])